In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier,  HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import RandomizedSearchCV
import warnings 
import missingno as msno
import scipy.stats as stats
from scipy.stats import chi2_contingency
import keras
from keras.models import Sequential
from keras.models import Model
from keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout
from scikeras.wrappers import KerasClassifier
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')



In [2]:
features = pd.read_csv('train_values.csv')
labels = pd.read_csv('train_labels.csv')

df= pd.concat([features, labels['damage_grade']], axis=1)
df.head()

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,3
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,2
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,3
3,590882,22,418,10694,2,10,6,5,t,r,...,0,0,0,0,0,0,0,0,0,2
4,201944,11,131,1488,3,30,8,9,t,r,...,0,0,0,0,0,0,0,0,0,3


In [3]:
#Splitting the data between training and testing
X_train, X_val, y_train, y_val = train_test_split(df.drop('damage_grade', axis=1), df['damage_grade'], random_state=20, shuffle=True, test_size=0.01)

print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(257994, 39)
(2607, 39)
(257994,)
(2607,)


In [4]:
n_rows, n_cols = X_train.shape

print(f"this dataset has {n_rows} rows and {n_cols} columns")
     

this dataset has 257994 rows and 39 columns


In [5]:
X_train.dtypes

building_id                                int64
geo_level_1_id                             int64
geo_level_2_id                             int64
geo_level_3_id                             int64
count_floors_pre_eq                        int64
age                                        int64
area_percentage                            int64
height_percentage                          int64
land_surface_condition                    object
foundation_type                           object
roof_type                                 object
ground_floor_type                         object
other_floor_type                          object
position                                  object
plan_configuration                        object
has_superstructure_adobe_mud               int64
has_superstructure_mud_mortar_stone        int64
has_superstructure_stone_flag              int64
has_superstructure_cement_mortar_stone     int64
has_superstructure_mud_mortar_brick        int64
has_superstructure_c

In [6]:
column_names = list(df.drop('damage_grade', axis=1).columns)
multi_class_columns = [col for col in column_names if df[col].dtype=='object' and df[col].nunique()>2]
binary_columns = [col for col in column_names if df[col].dtype=='object' and df[col].nunique()==2]
num_columns = [col for col in column_names if df[col].dtype!='object']
print(f"Multi Class columns: {multi_class_columns}")
print(f"Binary columns: {binary_columns}")
print(f"Numerical columns: {num_columns}")

Multi Class columns: ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']
Binary columns: []
Numerical columns: ['building_id', 'geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id', 'count_floors_pre_eq', 'age', 'area_percentage', 'height_percentage', 'has_superstructure_adobe_mud', 'has_superstructure_mud_mortar_stone', 'has_superstructure_stone_flag', 'has_superstructure_cement_mortar_stone', 'has_superstructure_mud_mortar_brick', 'has_superstructure_cement_mortar_brick', 'has_superstructure_timber', 'has_superstructure_bamboo', 'has_superstructure_rc_non_engineered', 'has_superstructure_rc_engineered', 'has_superstructure_other', 'count_families', 'has_secondary_use', 'has_secondary_use_agriculture', 'has_secondary_use_hotel', 'has_secondary_use_rental', 'has_secondary_use_institution', 'has_secondary_use_school', 'has_secondary_use_industry', 'has_secondary_use_health_post', '

In [7]:
# 3) ZERO-BASE THE TARGET (1→0, 2→1, 3→2)
df['damage_grade'] = df['damage_grade']-1

# 4) SPLIT INTO TRAIN/VAL
X = df.drop('damage_grade', axis=1)
y = df['damage_grade']   # now contains only 0, 1, 2

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
def get_pipeline(model):
  pipeline = Pipeline([
      ('preprocessor', ColumnTransformer(
          transformers=[
              ('cat', OneHotEncoder(), multi_class_columns), 
              ('num', make_pipeline(
                    FunctionTransformer(lambda x: x.drop(['building_id', 'count_families'], axis=1)),
                    MinMaxScaler()
                ), num_columns),
          ]
      )),
      ('model', model)
  ])
  return pipeline

In [9]:
import pandas as pd

# preprocessing & pipeline
from sklearn.compose      import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline     import Pipeline

# models & evaluation
from sklearn.linear_model     import LogisticRegression
from sklearn.naive_bayes      import GaussianNB
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier, AdaBoostClassifier, StackingClassifier
from lightgbm                 import LGBMClassifier
from sklearn.model_selection  import StratifiedKFold, cross_validate

# 1) — Load & split your training data, dropping the ID column up front
df_train = pd.read_csv('train_values.csv')
X_train  = df_train.drop(columns=['building_id'])
y_train  = pd.read_csv('train_labels.csv')['damage_grade']

# 2) — Load & prep your test set
df_test = pd.read_csv('test_values.csv')
ids     = df_test['building_id'].values
X_test  = df_test.drop(columns=['building_id'])

# 3) — Identify numeric vs. binary vs. multi-class categorical columns
num_cols   = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols   = X_train.select_dtypes(include=['object','category']).columns.tolist()
binary_cols      = [c for c in cat_cols if X_train[c].nunique() == 2]
multi_class_cols = [c for c in cat_cols if X_train[c].nunique() > 2]

# 4) — Build your ColumnTransformer
preprocessor = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), multi_class_cols),
    ('oe',  OrdinalEncoder(),                      binary_cols),
    ('std', StandardScaler(),                      num_cols)
])

# 5) — Helper to wrap any model in the pipeline
def get_pipeline(model):
    return Pipeline([
        ('pre', preprocessor),
        ('clf', model)
    ])

# 6) — Your list of base estimators
models = [
    LogisticRegression(random_state=42, max_iter=1000),
    GaussianNB(),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    AdaBoostClassifier(random_state=42),
    LGBMClassifier(random_state=42)
]

# 7) — Quick 3-fold stratified CV with accuracy and weighted F1
cv       = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring  = {'accuracy': 'accuracy', 'weighted_f1': 'f1_weighted'}
records  = []

for model in models:
    pipe       = get_pipeline(model)
    cv_results = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    records.append({
        'Model':     type(model).__name__,
        'Accuracy':  cv_results['test_accuracy'].mean(),
        'F1 score':  cv_results['test_weighted_f1'].mean()
    })

results_df = pd.DataFrame(records)
print("Cross-validation results:\n", results_df)

# 8) — Build a stacking ensemble of all base models
estimators = [(type(m).__name__, m) for m in models]
stacked_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=cv,
    passthrough=False,
    n_jobs=-1
)
stacked_pipe = Pipeline([('pre', preprocessor), ('clf', stacked_clf)])

# 9) — Train each base model on the full training set, predict test, and save CSV
for model in models:
    name = type(model).__name__
    pipe = get_pipeline(model)
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    pd.DataFrame({
        'building_id': ids,
        'damage_grade': preds
    }).to_csv(f'{name}_submission.csv', index=False)
    print(f'Wrote {name}_submission.csv')

# 10) — Finally, train & save the stacked model’s submission
stacked_pipe.fit(X_train, y_train)
final_preds = stacked_pipe.predict(X_test)

pd.DataFrame({
    'building_id': ids,
    'damage_grade': final_preds
}).to_csv('stacked_.csv', index=False)
print('Wrote stacked_submission.csv')


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.047554 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 833
[LightGBM] [Info] Number of data points in the train set: 173734, number of used features: 66
[LightGBM] [Info] Start training from score -2.339127
[LightGBM] [Info] Start training from score -0.564033
[LightGBM] [Info] Start training from score -1.094586
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043049 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 833
[LightGBM] [Info] Number of data points in the train set: 173734, number of used features: 66
[LightGBM] [Info] Start training from score -2.339187
[LightGBM] [Info] Start training from score -0.564023
[LightGBM] [Info] Start 

/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.090644 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 841
[LightGBM] [Info] Number of data points in the train set: 260601, number of used features: 68
[LightGBM] [Info] Start training from score -2.339167
[LightGBM] [Info] Start training from score -0.564030
[LightGBM] [Info] Start training from score -1.094580


/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/opt/apps/software/lang/Anaconda3/2024.02-1/lib/python3.11/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.117613 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 833
[LightGBM] [Info] Number of data points in the train set: 173734, number of used features: 66
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.102502 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 833
[LightGBM] [Info] Start training from score -2.339127
[LightGBM] [Info] Number of data points in the train set: 173734, number of used features: 66
[LightGBM] [Info] Start training from score -0.564033
[LightGBM] [Info] Start training from score -1.094586
[LightGBM] [Info] Start training from score -2.339187
[LightGBM] [Info] Start training from score -0.564023
[LightGBM] [Info] Start 

In [10]:
# for model in models:
#     name = type(model).__name__
#     pipeline = get_pipeline(model)
#     pipeline.fit(X_train, y_train)
#     preds = pipeline.predict(X_test)

#     print(f"{name:20s} →   ids: {len(ids):4d},  preds: {len(preds):4d}")

In [11]:
# import pandas as pd

# # ─── User‐replace this with the data you actually want predictions on ───
# # it could be X_val, X_test, or even X_train itself
# data_to_predict = X_val

# # build an empty DataFrame with the same index
# idx = data_to_predict.index if hasattr(data_to_predict, 'index') else range(len(data_to_predict))
# preds_df = pd.DataFrame(index=idx)

# # loop just like before—but this time fit+predict
# for model in models:
#     pipe = get_pipeline(model)
#     pipe.fit(X_train, y_train)                          # train on all of X_train
#     preds = pipe.predict(data_to_predict)               # make predictions
#     preds_df[type(model).__name__] = preds              # store in column

# # write a single CSV with every model’s column of predictions
# out_path = 'model_predictions.csv'
# preds_df.to_csv(out_path, index=True)
# print(f"Saved predictions CSV to {out_path}")
